# PCB Router — Round-Robin v3 (MaskablePPO)

Branch **`round-robin-v2`** of [`adikeshn/pcb-router-world`](https://github.com/adikeshn/pcb-router-world).

Run top to bottom: setup → W&B login → hyperparameters → **board preview** →
**acceptance checks** → train → results.

GPU is optional. Env stepping (the main cost) is CPU-bound; a T4 speeds up
PPO updates only.


## 1 · Setup


In [ ]:
BRANCH = 'round-robin-v2'
REPO   = 'https://github.com/adikeshn/pcb-router-world.git'

import os, sys
if not os.path.exists('pcb-router-world'):
    !git clone --branch $BRANCH --single-branch $REPO
%cd pcb-router-world
!pip -q install -r requirements.txt
sys.path.insert(0, os.getcwd())
print('setup complete')


## 2 · Weights & Biases login


In [ ]:
import wandb
wandb.login()


## 3 · Hyperparameters

Maps 1:1 to `pcb_router_rr2.config.Config`. Unknown keys raise immediately.

**Reward note:** every dense term is a TOTAL per episode, divided internally by
the episode's round/step count. This keeps the dense/terminal balance constant
across the whole budget range — previously dense grew with episode length while
terminal stayed fixed, which inverted the objective at long budgets.


In [ ]:
HPARAMS = dict(
    # ---------------- board (mm) ----------------
    board_width_mm   = 200.0,
    board_height_mm  = 150.0,
    edge_clearance_mm= 2.0,
    connector_rect   = (78.0, 38.0, 122.0, 50.0),
    # One pin per pad pair, ON the connector edge, separated beyond
    # trace clearance (both rules validated).
    pins = [
        (83.5, 50.0), (89.0, 50.0), (94.5, 50.0), (100.0, 50.0),
        (105.5, 50.0), (111.0, 50.0), (116.5, 50.0),   # 7 on top edge
        (85.0, 38.0), (100.0, 38.0), (115.0, 38.0),    # 3 on bottom edge
    ],
    obstacles = [],

    # ---------------- clearances (mm) ----------------
    trace_clearance_mm    = 1.33,
    self_clearance_mm     = 0.60,   # MUST be < step_mm (validated)
    obstacle_clearance_mm = 1.00,

    # ---------------- growth ----------------
    step_mm       = 1.0,
    budget_min_mm = 75.0,
    budget_max_mm = 110.0,
    ban_reverse   = True,
    use_breakout  = False,   # agent grows directly from the pins

    # ---------------- dense reward (per-episode TOTALS) ----------------
    spacing_dense_total     = 0.80,
    dense_spacing_target_mm = 16.0,
    path_penalty_total      = 0.25,
    path_soft_mm            = 4.0,
    self_penalty_total      = 0.25,   # NEW: penalises a trace coiling on itself
    self_soft_mm            = 4.0,
    turn_penalty_total      = 0.30,   # NEW: makes clean meanders beat jitter
    edge_penalty_total      = 0.20,
    edge_soft_mm            = 8.0,

    # ---------------- terminal reward ----------------
    #   terminal = base
    #            + spacing_reward_per_mm * min_endpoint_spacing_mm   (UNCAPPED)
    #            + w_path_clearance_bonus * capped clearance quality
    #            + w_endpoint_edge_bonus  * capped edge quality
    w_terminal_base        = 5.0,   # flat, just for finishing validly
    spacing_reward_per_mm  = 0.30,  # UNCAPPED linear: reward points per mm.
                                    # Endpoint spacing is a MINIMUM over pairs,
                                    # so every extra mm is a real improvement to
                                    # the worst pair -- no ceiling, no saturation.
    w_path_clearance_bonus = 1.50,  # capped: a clipped MEAN; uncapping it would
    terminal_clearance_target_mm = 14.0,  # reward traces sitting in open space
    w_endpoint_edge_bonus  = 1.00,  # capped: past probe reach, further is worth
    terminal_edge_target_mm = 15.0,       # nothing and would fight spacing
    endpoint_spec_mm       = 13.0,  # label + ranking tier only

    # ---------------- portfolio ----------------
    # bands x per_band = 5 x 5 = 25 layouts.
    # Stratification gives variety across LENGTH; the diversity filter runs
    # WITHIN each band to give variety across SHAPE.
    portfolio_bands    = 5,
    portfolio_per_band = 5,
    portfolio_stratify_by_budget = True,
    min_moved_frac     = 0.5,   # >= this frac of endpoints must move...
    min_point_shift_mm = 13.0,  # ...by >= this vs every entry in the same band

    # ---------------- PPO ----------------
    total_timesteps = 4_000_000,
    n_envs        = 8,
    seed          = 0,
    learning_rate = 3e-4,
    n_steps       = 512,
    batch_size    = 512,
    n_epochs      = 6,
    # 1/(1-gamma) must cover the episode length (n_traces x budget).
    # 10 x 110 = 1100 steps -> 0.999 gives a 1000-step horizon.
    gamma         = 0.999,
    gae_lambda    = 0.95,
    ent_coef      = 0.01,
    clip_range    = 0.2,
    net_arch      = [256, 256],
    device        = 'auto',

    # ---------------- exploration / eval / logging ----------------
    explorer_every_episodes = 200,
    explorer_episodes       = 5,
    explorer_momentum       = 0.95,
    eval_every_steps        = 100_000,
    eval_episodes           = 8,
    render_every_episodes   = 250,
    log_every_episodes      = 10,

    wandb_project  = 'pcb-routing',
    wandb_run_name = None,
    wandb_mode     = 'online',
)

from pcb_router_rr2.config import Config
cfg = Config().override(**HPARAMS)
cfg.validate()
print(f'config OK — {cfg.n_traces} traces')

import torch
dev = 'cuda (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'cpu'
print(f'PPO updates will run on: {dev}')
steps = cfg.n_traces * int(cfg.budget_max_mm)
print(f'longest episode: {steps} steps | planning horizon 1/(1-gamma) = {1/(1-cfg.gamma):.0f} steps')


## 4 · Board preview — check the layout BEFORE training


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from pcb_router_rr2.board import Board
from pcb_router_rr2.rendering import preview_figure

print(Board.from_config(cfg).summary())
print()
fig = preview_figure(cfg, save_path='board_preview.png')
plt.show()


## 5 · Acceptance checks

* **self_crossing_check** — a tight fold that the old arc-length exemption
  window missed must now be caught, while straight runs and 90° corners stay legal.
* **zero_violation_check** — 200 masked-random episodes, zero audited violations.
* **reward_scale_check** — DISCOUNT-AWARE, evaluated at both ends of the budget
  range. The old raw-sum version passed a config where dense reward was worth
  ~1.8× terminal from the agent's actual point of view.

If any check fails, fix the config — do not train.


In [ ]:
from pcb_router_rr2.validate import run_all
assert run_all(cfg), 'Acceptance checks FAILED — do not train with this config.'


## 6 · Train

Panels worth watching:
* `train/budget_mm_all` vs `train/budget_mm_gated` — if gated sits well below
  all, long budgets are failing the gate (a policy problem) rather than losing
  the ranking (a scoring problem).
* `eval_by_budget/gate_pass_*` — per-budget breakdown of the same question.
* `train/r_spacing` — UNCAPPED, so it should keep climbing all run; a plateau
  means the policy has stopped improving spacing, not that a cap was hit.
* `train/q_clear`, `train/q_edge` — capped terms. Pinning at 1.0 means that
  term has saturated and stopped providing gradient; raise its target.
* `portfolio/band*_count` and `band*_best_spacing` — which budget bands are
  filling and how good each length's best layout is.
* `train/min_self_gap_mm`, `train/turn_rate` — coiling tightness and jitter.
* `reward/*` — per-term decomposition.
* `portfolio/band*_terminal` — which budget bands are filled.

Interrupting the cell is safe: model and portfolio are saved.


In [ ]:
from pcb_router_rr2.train import train
run_dir = train(cfg)
print('artifacts in', run_dir)


## 7 · Results — portfolio (bands x slots)


In [ ]:
import json
from IPython.display import Image, display

with open(f'{run_dir}/portfolio/portfolio.json') as f:
    index = json.load(f)

print(f'{len(index)} entries ({cfg.portfolio_bands} budget bands x {cfg.portfolio_per_band} slots)')
for e in index:
    print(f"  rank {e['rank']} | band {e['budget_band']} {e['band_range_mm']} mm | "
          f"budget {e['budget_mm']:.0f}mm | spec={'PASS' if e['meets_spec'] else 'miss'} | "
          f"terminal={e['reward_terminal']:.2f} | spacing={e['min_endpoint_spacing_mm']:.1f}mm | "
          f"self-gap={e['min_self_distance_mm']:.2f}mm | turn rate={e['turn_rate']:.2f}")
    display(Image(e['png'], width=560))

### Optional: resume, or download everything
```python
run_dir = train(cfg, resume_model=f'{run_dir}/model_final.zip')
```
```python
!zip -r results.zip {run_dir}
from google.colab import files; files.download('results.zip')
```
